# 06. Support Vector Machines

Linear soft-margin SVM (`LinearSVC`). No probabilities (decision_function).
Tune regularization `C` with stratified CV (F1); keep dual=False.

In [1]:
import sys
sys.path.append("..")

from sklearn.svm import LinearSVC

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult("train"))
X_test, y_test = split_features_target(load_adult("test"))
print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [2]:
candidates = [0.01, 0.1, 1, 10]
selection_metric = "f1"

cv_by_C = {}
best_C = None
best_score = -1.0

for C in candidates:
    model = LinearSVC(
        C=C,
        dual=False,
        max_iter=5000,
        random_state=42,
    )

    pipe = make_model_pipeline(model)
    cv = cross_val_scores(pipe, X, y)
    cv_by_C[C] = cv
    score = cv[selection_metric]["mean"]
    print(f"C={C} CV {selection_metric}={score:.4f} (±{cv[selection_metric]['std']:.4f})")

    if score > best_score:
        best_score = score
        best_C = C

print("selected C:", best_C)

C=0.01 CV f1=0.6433 (±0.0080)
C=0.1 CV f1=0.6458 (±0.0088)
C=1 CV f1=0.6466 (±0.0087)
C=10 CV f1=0.6465 (±0.0087)
selected C: 1


In [3]:
final_model = LinearSVC(
    C=best_C,
    dual=False, 
    max_iter=5000, 
    random_state=42
    )

cv = cv_by_C[best_C]
pipe2 = make_model_pipeline(final_model)
test = evaluate_on_test(pipe2, X, y, X_test, y_test)

path = save_results(
    "svm_linear",
    cv,
    test,
    extra={
        "kernel": "linear",
        "C": best_C,
        "dual": False,
        "max_iter": 5000,
        "candidates": candidates,
        "selection_metric": selection_metric,
        "cv_by_C": {
            str(c): {m: v["mean"] for m, v in summary.items()}
            for c, summary in cv_by_C.items()
        },
    },
)

print("Test:", test["metrics"])
print("Confustion matrix:", test["confusion_matrix"])
print("saved:", path)


Test: {'accuracy': 0.8482279958233524, 'balanced_accuracy': 0.7582267102505572, 'f1': 0.6465455585753112, 'roc_auc': 0.8985614116965162, 'average_precision': 0.736556665221906}
Confustion matrix: [[11550, 885], [1586, 2260]]
saved: /Users/kang/Desktop/ML project/Estimating-Income/results/svm_linear.json
